In [ ]:
%cd /content

# remove old folder if it already exists
!rm -rf scperturbevalv2

# clone repo
!git clone https://github.com/nainika0402-creator/scperturbevalv2.git

# enter correct folder
%cd scperturbevalv2

# install package
%pip install -e .

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Metrics

In [ ]:
%%bash
set -euo pipefail
OUT_ROOT="/content/drive/MyDrive/scdfm_results"
CKPT_ROOT="${OUT_ROOT}/additive_ckpts"
COND_COL="perturbation"
CTRL_LABEL="control"

for FOLD in 0 1 2 3; do
  REAL="${CKPT_ROOT}/fold${FOLD}/real.h5ad"
  PRED="${CKPT_ROOT}/fold${FOLD}/pred.h5ad"
  OUT="${OUT_ROOT}/fold${FOLD}_mset1.csv"

  python -m scPerturbEval.evaluations \
    --real "$REAL" --pred "$PRED" \
    --condition-column "$COND_COL" --control-label "$CTRL_LABEL" \
    --space raw \
    --metrics root_mean_squared_error pearson_distance pcc_delta top_deg_recall \
    --deg-selection topn --top-n-degs 100 \
    --out "$OUT"
done

In [ ]:
%%bash
set -euo pipefail
OUT_ROOT="/content/drive/MyDrive/scdfm_results"
CKPT_ROOT="${OUT_ROOT}/additive_ckpts"
COND_COL="perturbation"
CTRL_LABEL="control"

for FOLD in 0 1 2 3; do
  REAL="${CKPT_ROOT}/fold${FOLD}/real.h5ad"
  PRED="${CKPT_ROOT}/fold${FOLD}/pred.h5ad"
  OUT="${OUT_ROOT}/fold${FOLD}_mset2.csv"

  python -m scPerturbEval.evaluations \
    --real "$REAL" --pred "$PRED" \
    --condition-column "$COND_COL" --control-label "$CTRL_LABEL" \
    --space raw \
    --metrics deg_direction_agreement deg_spearman_lfc pds_cosine matrix_distance \
    --deg-selection topn --top-n-degs 100 \
    --out "$OUT"
done

In [ ]:
%%bash
set -euo pipefail
OUT_ROOT="/content/drive/MyDrive/scdfm_results"
CKPT_ROOT="${OUT_ROOT}/additive_ckpts"
COND_COL="perturbation"
CTRL_LABEL="control"

for FOLD in 0 1 2 3; do
  REAL="${CKPT_ROOT}/fold${FOLD}/real.h5ad"
  PRED="${CKPT_ROOT}/fold${FOLD}/pred.h5ad"
  OUT="${OUT_ROOT}/fold${FOLD}_mset3.csv"

  python -m scPerturbEval.evaluations \
    --real "$REAL" --pred "$PRED" \
    --condition-column "$COND_COL" --control-label "$CTRL_LABEL" \
    --space raw \
    --metrics wmse pearson_delta_pert weighted_r2_delta \
    --pathway-gene-sets MSigDB_Hallmark_2020 \
    --pathway-top-k 10 \
    --pathway-reference perturbed_centroid \
    --out "$OUT"
done

In [ ]:
%%bash
set -euo pipefail
OUT_ROOT="/content/drive/MyDrive/scdfm_results"
CKPT_ROOT="${OUT_ROOT}/additive_ckpts"
COND_COL="perturbation"

for FOLD in 0 1 2 3; do
  REAL="${CKPT_ROOT}/fold${FOLD}/real.h5ad"
  PRED="${CKPT_ROOT}/fold${FOLD}/pred.h5ad"
  OUT="${OUT_ROOT}/fold${FOLD}_mset5_pca.csv"

  python -m scPerturbEval.evaluations \
    --real "$REAL" --pred "$PRED" \
    --condition-column "$COND_COL" \
    --space pca --n-components 50 \
    --metrics wasserstein mmd \
    --out "$OUT"
done

# Aggregating Metrics across 4 folds

In [ ]:
%%bash
set -euo pipefail
python - <<'PY'
import pandas as pd
from pathlib import Path

OUT_ROOT = Path("/content/drive/MyDrive/scdfm_results")
FOLDS = [0,1,2,3]

rows = []
for f in FOLDS:
    files = [
        OUT_ROOT / f"fold{f}_mset1.csv",
        OUT_ROOT / f"fold{f}_mset2.csv",
        OUT_ROOT / f"fold{f}_mset3.csv",
        OUT_ROOT / f"fold{f}_mset4_raw.csv",
        OUT_ROOT / f"fold{f}_mset5_pca.csv",
    ]
    s = None
    for fp in files:
        df = pd.read_csv(fp).iloc[0]
        s = df if s is None else s.combine_first(df)
    s["fold"] = f
    rows.append(s)

all_df = pd.DataFrame(rows)
all_df.to_csv(OUT_ROOT / "scdfm_all_metrics_fold_rows.csv", index=False)

num_cols = [c for c in all_df.columns if c not in ["space","fold"] and pd.api.types.is_numeric_dtype(all_df[c])]
agg = pd.DataFrame({
    "metric": num_cols,
    "mean": [all_df[c].mean() for c in num_cols],
    "std":  [all_df[c].std() for c in num_cols],
})
agg["mean_std"] = agg["mean"].map(lambda x: f"{x:.6g}") + " ± " + agg["std"].map(lambda x: f"{x:.3g}")
agg.to_csv(OUT_ROOT / "scdfm_all_metrics_aggregated.csv", index=False)

print("Wrote:", OUT_ROOT / "scdfm_all_metrics_fold_rows.csv")
print("Wrote:", OUT_ROOT / "scdfm_all_metrics_aggregated.csv")
PY